# DAIC-WOZ eGeMAPS exploratory analysis

This adapts the TORGO notebook's data checks, feature-distribution plots, PCA, and random-forest analysis to DAIC-WOZ. Each row is one participant-answer segment's eGeMAPSv02 functionals. The model trains on the official train split and evaluates on the separate dev split, so participants are not mixed across train and test. This is exploratory analysis, not a clinical assessment.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

sns.set_theme(style="whitegrid")
TRAIN_CSV_PATH = Path("data/processed/daicwoz_train_egemaps.csv")
DEV_CSV_PATH = Path("data/processed/daicwoz_dev_egemaps.csv")
if not TRAIN_CSV_PATH.exists():
    TRAIN_CSV_PATH = Path("../data/processed/daicwoz_train_egemaps.csv")
if not DEV_CSV_PATH.exists():
    DEV_CSV_PATH = Path("../data/processed/daicwoz_dev_egemaps.csv")
for csv_path in (TRAIN_CSV_PATH, DEV_CSV_PATH):
    if not csv_path.is_file():
        raise FileNotFoundError(
            f"Could not find {csv_path}. Extract both train and dev features first."
        )

train_df = pd.read_csv(TRAIN_CSV_PATH)
dev_df = pd.read_csv(DEV_CSV_PATH)
train_df["split"] = "train"
dev_df["split"] = "dev"
df = pd.concat([train_df, dev_df], ignore_index=True)
df.head()

## 1. Check the extracted data

The number of feature columns can vary with openSMILE versions/configuration. Check the actual shape and missing values rather than assuming a fixed column count.

In [ ]:
metadata_cols = {"participant_id", "audio_file", "label"}
feature_cols = [
    column for column in df.select_dtypes(include=np.number).columns
    if column not in metadata_cols
]
print("Shape:", df.shape)
print("Participants:", df["participant_id"].nunique())
print("eGeMAPS feature count:", len(feature_cols))
print("\nMissing values by column (nonzero only):")
print(df.isna().sum().loc[lambda counts: counts > 0])
if "label" in df:
    print("\nClass distribution (0 = below threshold, 1 = at/above threshold):")
    print(df["label"].value_counts(dropna=False).sort_index())

if not feature_cols:
    raise ValueError("No numeric eGeMAPS feature columns were found.")

## 2. Visualize selected feature distributions

Compare a few familiar pitch, loudness, voice-quality, and spectral features across the binary PHQ-8 label. Only columns that exist in this openSMILE output are plotted.

In [ ]:
preferred_features = [
    "F0semitoneFrom27.5Hz_sma3nz_amean",
    "loudness_sma3_amean",
    "jitterLocal_sma3nz_amean",
    "shimmerLocaldB_sma3nz_amean",
    "HNRdBACF_sma3nz_amean",
    "alphaRatioV_sma3nz_amean",
]
plot_features = [name for name in preferred_features if name in feature_cols]
if not plot_features:
    plot_features = feature_cols[: min(6, len(feature_cols))]

if "label" not in df or df["label"].dropna().nunique() < 2:
    print("Two labeled classes are required for class-wise plots.")
else:
    plot_data = df.dropna(subset=["label"])
    fig, axes = plt.subplots(
        nrows=(len(plot_features) + 1) // 2,
        ncols=2,
        figsize=(14, 4 * ((len(plot_features) + 1) // 2)),
        squeeze=False,
    )
    for ax, feature in zip(axes.flat, plot_features):
        sns.histplot(data=plot_data, x=feature, hue="label", kde=True, ax=ax)
        ax.set_title(feature)
    for ax in axes.flat[len(plot_features):]:
        ax.remove()
    plt.tight_layout()
    plt.show()

## 3. PCA visualization

Standardize the features before projecting them to two dimensions. Median imputation keeps rows with a small number of missing feature values available for visualization.

In [ ]:
from sklearn.decomposition import PCA
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

pca_data = df.dropna(subset=["label"]).copy()
if pca_data["label"].nunique() >= 2 and len(feature_cols) >= 2:
    X_pca = SimpleImputer(strategy="median").fit_transform(pca_data[feature_cols])
    X_pca = StandardScaler().fit_transform(X_pca)
    pca = PCA(n_components=2, random_state=42)
    coordinates = pca.fit_transform(X_pca)
    pca_data["PC1"] = coordinates[:, 0]
    pca_data["PC2"] = coordinates[:, 1]
    print("Variance explained by first two components:", pca.explained_variance_ratio_.sum())
    sns.scatterplot(data=pca_data, x="PC1", y="PC2", hue="label", alpha=0.8)
    plt.title("PCA of eGeMAPS features")
    plt.show()
else:
    print("Need at least two labeled classes and two numeric features for PCA.")

## 4. Random forest: official train/dev evaluation

The CSVs contain multiple segments per participant. Train only on the official train feature file and evaluate on the official dev feature file. Verify that participant IDs do not overlap; otherwise stop rather than reporting a leaked evaluation. The number of participants is limited, so treat these metrics as exploratory.

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import balanced_accuracy_score, classification_report, confusion_matrix
from sklearn.pipeline import make_pipeline

N_JOBS = 8  # kedi is shared; never use every core

train_model_data = train_df.dropna(subset=["label", "participant_id"]).copy()
dev_model_data = dev_df.dropna(subset=["label", "participant_id"]).copy()
train_model_data["label"] = pd.to_numeric(train_model_data["label"], errors="raise").astype(int)
dev_model_data["label"] = pd.to_numeric(dev_model_data["label"], errors="raise").astype(int)
train_people = set(train_model_data["participant_id"].astype(str))
dev_people = set(dev_model_data["participant_id"].astype(str))
overlap = train_people & dev_people
if overlap:
    raise ValueError(f"Train/dev participant overlap ({len(overlap)} IDs); refusing leaked evaluation.")
if train_model_data["label"].nunique() != 2 or dev_model_data["label"].nunique() != 2:
    raise ValueError("Both train and dev need labeled examples from both classes (0 and 1).")

X_train = train_model_data[feature_cols]
X_test = dev_model_data[feature_cols]
y_train = train_model_data["label"]
y_test = dev_model_data["label"]
print("Train rows/participants:", len(train_model_data), len(train_people))
print("Dev rows/participants:", len(dev_model_data), len(dev_people))
print("Train class counts:\n", y_train.value_counts().sort_index())
print("Test class counts:\n", y_test.value_counts().sort_index())

model = make_pipeline(
    SimpleImputer(strategy="median"),
    RandomForestClassifier(
        n_estimators=300,
        class_weight="balanced",
        random_state=42,
        n_jobs=N_JOBS,
    ),
)
model.fit(X_train, y_train)
predictions = model.predict(X_test)
print("\nBalanced accuracy:", balanced_accuracy_score(y_test, predictions))
print("\n", classification_report(
    y_test,
    predictions,
    labels=[0, 1],
    target_names=["below threshold", "at/above threshold"],
    zero_division=0,
))
sns.heatmap(
    confusion_matrix(y_test, predictions, labels=[0, 1]),
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=["below threshold", "at/above threshold"],
    yticklabels=["below threshold", "at/above threshold"],
)
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Official dev-split confusion matrix")
plt.show()

## 5. Feature importance

Random-forest impurity importances are a quick descriptive summary. Correlated acoustic features can split importance across columns, so do not interpret this ranking as causal evidence.

In [ ]:
forest = model.named_steps["randomforestclassifier"]
importance = pd.Series(forest.feature_importances_, index=feature_cols)
print(importance.nlargest(20).round(4).to_string())
importance.nlargest(20).sort_values().plot.barh(figsize=(9, 7))
plt.xlabel("Random forest feature importance")
plt.title("Top 20 eGeMAPS features")
plt.tight_layout()
plt.show()

## 6. Threshold-free and participant-level evaluation

Section 4 scores segments at a 0.5 probability cutoff. Two problems with that: the cutoff can hide a model that ranks people correctly but rarely crosses 0.5, and PHQ-8 labels belong to participants, not segments, so 1,862 dev segments are really 35 correlated groups. Here every model is scored with ROC-AUC at both levels. Participant-level scores average each person's segment probabilities. The 95% interval resamples dev participants (not segments), so it reflects how few people the dev split contains. An interval that includes 0.5 means no evidence of signal.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import StandardScaler


def bootstrap_auc_ci(y_true, scores, n_boot=2000, seed=42):
    """95% percentile interval for ROC-AUC, resampling rows (here: participants)."""
    y_true = np.asarray(y_true)
    scores = np.asarray(scores)
    rng = np.random.default_rng(seed)
    aucs = []
    for _ in range(n_boot):
        idx = rng.integers(0, len(y_true), len(y_true))
        if len(np.unique(y_true[idx])) == 2:
            aucs.append(roc_auc_score(y_true[idx], scores[idx]))
    return np.percentile(aucs, [2.5, 97.5])


def participant_scores(frame, segment_probs):
    """Average segment probabilities into one score and label per participant."""
    per_person = (
        frame[["participant_id", "label"]]
        .assign(prob=segment_probs)
        .groupby("participant_id")
        .agg(label=("label", "first"), prob=("prob", "mean"))
    )
    return per_person["label"].to_numpy(), per_person["prob"].to_numpy()


results = []


def record(name, segment_auc, person_labels, person_probs):
    low, high = bootstrap_auc_ci(person_labels, person_probs)
    results.append({
        "model": name,
        "segment AUC": segment_auc,
        "participant AUC": roc_auc_score(person_labels, person_probs),
        "participant AUC 95% CI": f"[{low:.2f}, {high:.2f}]",
        "participant balanced acc @0.5": balanced_accuracy_score(person_labels, (person_probs >= 0.5).astype(int)),
    })


# Random forest from section 4, re-scored.
rf_dev_probs = model.predict_proba(X_test)[:, 1]
record(
    "Random forest (segments)",
    roc_auc_score(y_test, rf_dev_probs),
    *participant_scores(dev_model_data, rf_dev_probs),
)

## 7. Simpler models that are harder to overfit to speakers

With only 107 training speakers, a deep random forest can memorize each speaker's voice instead of learning anything about PHQ-8 status. Two lower-capacity alternatives:

- **Segment-level logistic regression:** standardized features with strong L2 regularization.
- **Participant-level models:** collapse each person's segments into the mean and standard deviation of every feature, giving one row per person (107 train, 35 dev). This matches the level at which labels are defined and removes repeated rows from the same speaker entirely.

Within-speaker z-scoring is deliberately not used: because every segment from a person shares one label, it would erase exactly the between-person differences these models need to detect.

In [ ]:
logreg = make_pipeline(
    SimpleImputer(strategy="median"),
    StandardScaler(),
    LogisticRegression(C=0.01, class_weight="balanced", max_iter=5000),
)
logreg.fit(X_train, y_train)
lr_dev_probs = logreg.predict_proba(X_test)[:, 1]
record(
    "Logistic regression (segments)",
    roc_auc_score(y_test, lr_dev_probs),
    *participant_scores(dev_model_data, lr_dev_probs),
)


def participant_features(frame):
    """One row per participant: mean and std of each feature over their segments."""
    grouped = frame.groupby("participant_id")
    features = grouped[feature_cols].agg(["mean", "std"])
    features.columns = [f"{name}__{stat}" for name, stat in features.columns]
    return features, grouped["label"].first()


X_train_people, y_train_people = participant_features(train_model_data)
X_dev_people, y_dev_people = participant_features(dev_model_data)
print("Participant-level rows (train/dev):", len(X_train_people), len(X_dev_people))
print("Participant-level features:", X_train_people.shape[1])

participant_models = {
    "Logistic regression (per participant)": make_pipeline(
        SimpleImputer(strategy="median"),
        StandardScaler(),
        LogisticRegression(C=0.01, class_weight="balanced", max_iter=5000),
    ),
    "Random forest (per participant)": make_pipeline(
        SimpleImputer(strategy="median"),
        RandomForestClassifier(
            n_estimators=500,
            min_samples_leaf=3,
            class_weight="balanced",
            random_state=42,
            n_jobs=N_JOBS,
        ),
    ),
}
for name, person_model in participant_models.items():
    person_model.fit(X_train_people, y_train_people)
    person_probs = person_model.predict_proba(X_dev_people)[:, 1]
    # Segment AUC is undefined when the model only sees one row per person.
    record(name, np.nan, y_dev_people.to_numpy(), person_probs)

## 8. Summary

Compare participant AUC across models and check whether each interval excludes 0.5. With 35 dev participants, differences between models smaller than the interval widths are not meaningful.

In [ ]:
summary = pd.DataFrame(results).set_index("model")
print("Dev participants:", len(y_dev_people), "| positive:", int(y_dev_people.sum()))
print(summary.round(3).to_string())